# Sports & Leisure Revenue Decline Analysis
**Period:** 2018-07 to 2018-26  
**Dataset:** Olist E-Commerce (Brazil)  
**Goal:** Identify root causes of 51% revenue decline in Sports & Leisure category

## 1. Setup

In [30]:
import pandas as pd
import numpy as np

## 2. Load Data

In [31]:
xl = pd.ExcelFile(r'..\olist_dataset_clean.xlsx')

df_order_items = xl.parse('fact_order_items')
df_reviews = xl.parse('order_reviews')
df_customers = xl.parse('customers')
df_sellers = xl.parse('sellers')
df_products = xl.parse('products')
df_date = xl.parse('dim_date')


In [32]:
print('Duplicates check:')
print(f'  order_items:        {df_order_items.duplicated(subset="order_id").sum()}')
print(f'  products:           {df_products.duplicated(subset="product_key").sum()}')
print(f'  reviews:            {df_reviews.duplicated(subset="order_id").sum()}')
print(f'  sellers:            {df_sellers.duplicated(subset="seller_key").sum()}')
print(f'  customers:          {df_customers.duplicated(subset="customer_key").sum()}')

Duplicates check:
  order_items:        13984
  products:           0
  reviews:            0
  sellers:            0
  customers:          0


## 3. Build Base DataFrame

In [33]:
df_base = (
    df_order_items
    .merge(df_products, on='product_key')
    .merge(df_sellers, on='seller_key')
    .merge(df_customers, on='customer_key')
    .merge(df_date, left_on='order_date_key', right_on='Date', how='left')
    .merge(df_reviews, on='order_id', how='left')
)
# Filter: delivered orders only, exclude last 2 incomplete months
starting_month = df_base['Date'].dt.to_period('M').max() - 2
df_base = df_base[
    (df_base['Date'].dt.to_period('M') <= starting_month) &
    (df_base['order_status'] == 'delivered')
].copy()

In [34]:
sl = df_base[df_base['product_category_name_english'] == 'sports_leisure'].copy()

# Price buckets (Procentiliai - paliekam tavo gerą idėją)
low  = sl['price'].quantile(0.33)
high = sl['price'].quantile(0.66)
print(f'Price thresholds: low <= {low:.2f}, mid <= {high:.2f}, high > {high:.2f}')

def price_bucket(x):
    if x <= low:   return 'low'
    elif x <= high: return 'mid'
    else:           return 'high'

sl['price_category'] = sl['price'].apply(price_bucket)
sl['Delivery_Time']  = pd.to_numeric(sl['Delivery_Time'], errors='coerce')

Price thresholds: low <= 54.90, mid <= 109.95, high > 109.95


In [35]:
# 1. Dataset Filtering
sl = df_base[
    df_base["product_category_name_english"] == "sports_leisure"
].copy()
sl["Date"] = pd.to_datetime(sl["Date"])

# 2. Kainų segmentai naudojant procentilius (tavo originali idėja)
low = sl["price"].quantile(0.33)
high = sl["price"].quantile(0.66)
print(
    f"Price thresholds: low <= {low:.2f}, mid <= {high:.2f}, high > {high:.2f}"
)


Price thresholds: low <= 54.90, mid <= 109.95, high > 109.95


In [36]:
# 2. Price Segmentation (Percentiles)
low = sl["price"].quantile(0.33)
high = sl["price"].quantile(0.66)


def price_bucket(x):
    if x <= low:
        return "low"
    elif x <= high:
        return "mid"
    else:
        return "high"


sl["price_category"] = sl["price"].apply(price_bucket)
sl["Delivery_Time"] = pd.to_numeric(sl["Delivery_Time"], errors="coerce")

In [37]:
# 3. Weekly Base Aggregation
sl_export = (
    sl.groupby("Year Week")
    .agg(
        revenue=("price", "sum"),
        orders=("order_id", "nunique"),
        avg_review=("review_score", "mean"),
        avg_delivery=("Delivery_Time", "mean"),
        avg_freight_ratio=("freight_ratio", "mean"),
        unique_customers=("customer_unique_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
    )
    .sort_index()
)

sl_export["aov"] = sl_export["revenue"] / sl_export["orders"]

In [38]:
# 4. Revenue by Price Segment
price_segments = (
    sl.groupby(["Year Week", "price_category"])["price"]
    .sum()
    .unstack()
    .fillna(0)
)
price_segments = price_segments[["high", "low", "mid"]]
price_segments.columns = ["revenue_high", "revenue_low", "revenue_mid"]
sl_export = sl_export.join(price_segments)

In [39]:
# 5. Seller Cohort Classification (New vs Returning)
pre_s = sl[sl["Year Week"] < "2018-07"]
dur_s = sl[(sl["Year Week"] >= "2018-07") & (sl["Year Week"] <= "2018-26")]

pre_seller_ids = set(pre_s["seller_id"].unique())
dur_seller_ids = set(dur_s["seller_id"].unique())
returning_sellers = pre_seller_ids & dur_seller_ids


def seller_status(seller_id):
    if seller_id in returning_sellers:
        return "returning"
    else:
        return "new"


sl["seller_status"] = sl["seller_id"].apply(seller_status)

In [40]:
# 6. Revenue by Seller Cohort
seller_weekly = (
    sl.groupby(["Year Week", "seller_status"])["price"]
    .sum()
    .unstack()
    .fillna(0)
)
seller_weekly = (
    seller_weekly[["new", "returning"]]
    if "new" in seller_weekly.columns
    else seller_weekly
)
seller_weekly.columns = ["revenue_new_sellers", "revenue_returning_sellers"]
sl_export = sl_export.join(seller_weekly)

In [41]:
# 7. 6-Week Rolling Metrics Calculation
cols_to_roll = [
    "revenue",
    "orders",
    "aov",
    "avg_review",
    "avg_delivery",
    "avg_freight_ratio",
    "unique_customers",
    "unique_sellers",
    "revenue_high",
    "revenue_low",
    "revenue_mid",
    "revenue_new_sellers",
    "revenue_returning_sellers",
]

for col in cols_to_roll:
    sl_export[f"{col}_6w"] = (
        sl_export[col].rolling(window=6, min_periods=1).mean()
    )

In [42]:
# 8. Timeframe Filtering & Indexing
sl_export_final = sl_export.loc["2018-07":"2018-26"].copy()
sl_export_final["Sort Week"] = (
    sl_export_final.index.str.replace("-", "").astype(int)
)

In [43]:
# 9. File Export (EU Format Configuration)
sl_export_final.to_csv("sports_leisure_weekly.csv", index=True, decimal=",", sep=";")

print("Process complete. File exported: sports_leisure_weekly.csv")
sl_export_final.head()

Process complete. File exported: sports_leisure_weekly.csv


,revenue,orders,avg_review,avg_delivery,avg_freight_ratio,unique_customers,unique_sellers,aov,revenue_high,revenue_low,...,avg_delivery_6w,avg_freight_ratio_6w,unique_customers_6w,unique_sellers_6w,revenue_high_6w,revenue_low_6w,revenue_mid_6w,revenue_new_sellers_6w,revenue_returning_sellers_6w,Sort Week
Year Week,,,,,,,,,,,,,,,,,,,,,
2018-07,17234.08,135,4.141379,16.373368,0.267820,129,59,127.659852,12560.77,1322.78,...,14.917072,0.264104,135.166667,62.166667,14735.730000,1534.915000,4123.498333,1362.016667,19032.126667,201807
2018-08,15285.65,147,3.418182,17.429560,0.285235,146,56,103.984014,7882.10,1877.80,...,15.614938,0.261255,136.000000,61.166667,13105.305000,1635.866667,4141.476667,1130.568333,17752.080000,201808
2018-09,20977.93,168,3.606557,17.662801,0.266914,166,74,124.868631,14604.15,2094.64,...,16.220303,0.264366,140.333333,61.833333,13361.515000,1747.191667,4148.170000,1256.146667,18000.730000,201809
2018-10,24871.30,173,3.603352,17.645091,0.255353,171,69,143.764740,18669.33,1763.24,...,16.897527,0.261999,149.000000,62.333333,13678.331667,1774.470000,4415.336667,1040.923333,18827.215000,201810
2018-11,18439.68,147,3.824561,15.271912,0.296309,145,77,125.440000,12484.23,2120.64,...,16.724449,0.266495,148.500000,64.166667,13601.791667,1784.643333,4392.738333,1228.805000,18550.368333,201811
